In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import (
    load_labels,
    make_label_mapping,
    make_transforms,
    SignLanguageDataset,
    stratified_split,
)
from src.models import ImprovedMLP
from src.train_utils import evaluate

DATA_ROOT = PROJECT_ROOT / "sign_lang_train" / "sign_lang_train"

# Load labels and class mapping
df = load_labels(DATA_ROOT)
label_to_idx, idx_to_label = make_label_mapping(df["label"])

# Recreate the same split as during training
train_df, val_df, test_df = stratified_split(
    df,
    test_size=0.2,
    val_size=0.2,
    random_state=42,
)

# IMPORTANT: no augmentation for evaluation
eval_transform = make_transforms(image_size=64, augment=False)

train_dataset = SignLanguageDataset(
    train_df,
    label_to_idx=label_to_idx,
    transform=eval_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Same architecture as during improved training
model = ImprovedMLP(
    input_size=64 * 64,
    hidden_size=512,
    num_classes=len(label_to_idx),
    dropout=0.3,
).to(device)

model.load_state_dict(
    torch.load(
        PROJECT_ROOT / "saved_models" / "mlp_improved.pt",
        map_location=device,
    )
)

criterion = nn.CrossEntropyLoss()

train_result = evaluate(
    model=model,
    loader=train_loader,
    criterion=criterion,
    device=device,
)

print("Improved MLP train accuracy:", train_result["accuracy"])
print("Improved MLP train loss:", train_result["loss"])
print("Improved MLP train macro-F1:", train_result["macro_f1"])